# Case Linking - End to End Tests

Reconciles the post-publish leg of case linking:
- `case_link_payloads` (gold) - what we intended to send
- result/ack audit (`ack_audit`) - what the function app reported back from CCD

Together with 1_/2_/3_ this covers the pipeline end to end **up to "the function app reported the CCD call succeeded"**. It does NOT prove CCD persisted/displays the links - that needs the XUI check.

In [0]:
########################
# test Setup
########################

state_under_test = "caseLinking"
run_tag_default = "end_to_end"

dbutils.widgets.text("run_tag", run_tag_default)
run_tag = dbutils.widgets.get("run_tag") or run_tag_default

# Optional: scope publish + ack audits to a single pipeline RunID. Blank = all rows.
dbutils.widgets.text("target_run_id", "")
target_run_id = dbutils.widgets.get("target_run_id")

# Per-case link evidence: how many cases to print + record. None or 0 = ALL cases.
evidence_sample_size = 5

In [0]:
import os
import sys
import time as perf_time
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.functions import col, size, upper, when, max as smax
import inspect

run_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
test_results_path = "/Workspace/Users/" + run_user + "/Results/Case_Linking_Tests"
os.makedirs(test_results_path, exist_ok=True)

for _mod in list(sys.modules.keys()):
    if _mod.startswith("Test_Functions") or _mod == "models.test_result":
        del sys.modules[_mod]

from models.test_result import TestResult
from Test_Functions.report_helpers import (
    build_report_folder,
    write_csv, write_xlsx, write_html,
    count_by_status,
    create_run, create_results,
)

print(f"User: {run_user}")
print(f"state_under_test: {state_under_test}")
print(f"run_tag: {run_tag}")
print(f"target_run_id: {target_run_id or '(all runs)'}")
print(f"Results folder root: {test_results_path}")

In [0]:
#######################
# Spark Config (state-independent - runs once)
#######################
config_path = "dbfs:/configs/config.json"
config = spark.read.option("multiline", "true").json(config_path)
first_row = config.first()
env = first_row["env"].strip().lower()
env_name = env
lz_key = first_row["lz_key"].strip().lower()
keyvault_name = f"ingest{lz_key}-meta002-{env}"
KeyVault_name = keyvault_name

client_secret = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-TENANT-ID")
client_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-ID")

curated_storage    = f"ingest{lz_key}curated{env}"
checkpoint_storage = f"ingest{lz_key}xcutting{env}"
raw_storage        = f"ingest{lz_key}raw{env}"
landing_storage    = f"ingest{lz_key}landing{env}"
external_storage   = f"ingest{lz_key}external{env}"

for storage in (curated_storage, checkpoint_storage, raw_storage, landing_storage, external_storage):
    spark.conf.set(f"fs.azure.account.auth.type.{storage}.dfs.core.windows.net", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage}.dfs.core.windows.net", client_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage}.dfs.core.windows.net", client_secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print(f"env={env}  lz_key={lz_key}")

## Load tables

In [0]:
state_dir = "CASE_LINKING"

ack_audit_path = f"abfss://silver@ingest{lz_key}curated{env}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/{state_dir}/ack_audit"

silver_case_link_groups = spark.table("hive_metastore.ariadm_active_appeals.silver_case_link_groups")
case_link_combinations  = spark.table("hive_metastore.ariadm_active_appeals.case_link_combinations")
case_link_payloads      = spark.table("hive_metastore.ariadm_active_appeals.case_link_payloads")
ack_audit               = spark.read.format("delta").load(ack_audit_path)

if target_run_id:
    ack_audit = ack_audit.filter(col("RunID") == target_run_id)

print(f"silver_case_link_groups rows: {silver_case_link_groups.count():,}")
print(f"case_link_combinations rows:  {case_link_combinations.count():,}")
print(f"case_link_payloads rows:      {case_link_payloads.count():,}")
print(f"ack_audit rows:               {ack_audit.count():,}")

In [0]:
run_start_datetime = datetime.utcnow()
all_test_results = []
t0 = perf_time.time()

# Coverage - every payload reaches the ack audit; no orphan acks

In [0]:
def test_pipeline_coverage(payloads_df, ack_df):
    fn = inspect.stack()[0].function
    results = []
    try:
        payload_refs = payloads_df.select(col("CCDCaseReferenceNumberFrom").alias("ref")).distinct()
        ack_refs     = ack_df.select(col("CCDCaseReferenceNumber").alias("ref")).distinct()

        p_n = payload_refs.count()

        miss_ack = payload_refs.subtract(ack_refs)
        miss_ack_n = miss_ack.count()
        msg = f"expected: every payload reaches ack audit | actual: {miss_ack_n} of {p_n} payloads with no ack result"
        if miss_ack_n:
            msg += f" (sample: {[r.ref for r in miss_ack.limit(5).collect()]})"
        results.append(TestResult("case_link_e2e.payloads_acked",
                                  "PASS" if miss_ack_n == 0 else "FAIL", msg, state_under_test, fn))

        orphan = ack_refs.subtract(payload_refs)
        orphan_n = orphan.count()
        msg = f"expected: no ack rows without a payload | actual: {orphan_n} orphan ack refs"
        if orphan_n:
            msg += f" (sample: {[r.ref for r in orphan.limit(5).collect()]})"
        results.append(TestResult("case_link_e2e.no_orphan_acks",
                                  "PASS" if orphan_n == 0 else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult("case_link_e2e.coverage", "FAIL", f"Test crashed: {e}", state_under_test, fn))
    return results


all_test_results.extend(test_pipeline_coverage(case_link_payloads, ack_audit))

# Result status - terminal ack Status is SUCCESS for every case (allowing retries)

In [0]:
def test_ack_status_success(ack_df):
    """A case may have multiple ack rows (function-app retries). Reduce to a terminal
    status per CCD reference: SUCCESS if ANY attempt succeeded, else treated as failed."""
    fn = inspect.stack()[0].function
    results = []
    try:
        ack_term = (
            ack_df.groupBy("CCDCaseReferenceNumber")
                  .agg(smax(when(upper(col("Status")) == "SUCCESS", 1).otherwise(0)).alias("has_success"))
        )
        failed = ack_term.filter(col("has_success") == 0)
        failed_n = failed.count()
        total_n = ack_term.count()

        msg = f"expected: terminal ack Status SUCCESS for every case | actual: {failed_n} of {total_n} cases never succeeded"
        if failed_n:
            err_sample = (
                ack_df.join(failed.select("CCDCaseReferenceNumber"), "CCDCaseReferenceNumber")
                      .filter(upper(col("Status")) != "SUCCESS")
                      .select("CCDCaseReferenceNumber", "Error")
                      .limit(5).collect()
            )
            msg += " (sample ref,error: " + str([(r.CCDCaseReferenceNumber, (r.Error or "")[:500]) for r in err_sample]) + ")"
        results.append(TestResult("case_link_e2e.ack_status_success",
                                  "PASS" if failed_n == 0 else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult("case_link_e2e.ack_status_success", "FAIL", f"Test crashed: {e}", state_under_test, fn))
    return results


all_test_results.extend(test_ack_status_success(ack_audit))

# Count match - ack CaseLinkCount equals the number of links in the payload

In [0]:
def test_caselink_count_match(payloads_df, ack_df):
    fn = inspect.stack()[0].function
    results = []
    try:
        payload_counts = payloads_df.select(
            col("CCDCaseReferenceNumberFrom").alias("ref"),
            size("caseLinks").alias("payload_count"),
        )
        # ack CaseLinkCount is a string; take the max per ref (consistent across retries)
        ack_counts = (
            ack_df.groupBy("CCDCaseReferenceNumber")
                  .agg(smax(col("CaseLinkCount").cast("int")).alias("ack_count"))
        )

        joined = payload_counts.join(
            ack_counts, payload_counts.ref == ack_counts.CCDCaseReferenceNumber, "inner"
        )
        bad = joined.filter(col("payload_count") != col("ack_count"))
        bad_n = bad.count()
        checked = joined.count()

        msg = f"expected: ack CaseLinkCount == size(caseLinks) | actual: {bad_n} of {checked} mismatches"
        if bad_n:
            sample = [(r.ref, r.payload_count, r.ack_count)
                      for r in bad.select("ref", "payload_count", "ack_count").limit(5).collect()]
            msg += f" (sample ref,payload,ack: {sample})"
        results.append(TestResult("case_link_e2e.caselink_count_match",
                                  "PASS" if bad_n == 0 else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult("case_link_e2e.caselink_count_match", "FAIL", f"Test crashed: {e}", state_under_test, fn))
    return results


all_test_results.extend(test_caselink_count_match(case_link_payloads, ack_audit))

# Count funnel - trace link + case counts from silver through gold to the results audit

In [0]:
def test_count_funnel(silver_df, combinations_df, payloads_df, ack_df):
    """Trace counts through every stage so nothing is silently lost.

    LINKS:   expected (silver n(n-1)/2) -> combinations rows -> combinations with both
             CCD refs resolved -> payload entries -> ack links returned.
    CASES:   distinct payload 'from' cases -> distinct ack cases -> ack SUCCESS cases.

    Reductions at silver->combinations (same triple across groups collapses) and
    combinations->resolved (cases not migrated to CCD) are EXPECTED and reported as
    informational deltas. The hard 'nothing lost' equalities are:
      resolved combinations == payload links == ack links, and payload cases == ack cases.
    """
    fn = inspect.stack()[0].function
    results = []
    try:
        # ---- LINK-level ----
        per_linkno = (silver_df.filter(col("LinkNo").isNotNull())
                      .groupBy("LinkNo").agg(F.countDistinct("CaseNo").alias("n")))
        expected_links = (per_linkno
                          .withColumn("pairs", (col("n") * (col("n") - 1) / 2).cast("long"))
                          .agg(F.sum("pairs").alias("t")).first().t) or 0

        comb_total = combinations_df.count()
        comb_resolved = combinations_df.filter(
            col("CCDCaseReferenceNumberFrom").isNotNull() &
            col("CCDCaseReferenceNumberTo").isNotNull()
        ).count()

        payload_links = (payloads_df.select(F.sum(size("caseLinks")).alias("t")).first().t) or 0

        ack_term = (ack_df.groupBy("CCDCaseReferenceNumber")
                    .agg(smax(col("CaseLinkCount").cast("long")).alias("c")))
        ack_links = (ack_term.agg(F.sum("c").alias("t")).first().t) or 0

        # ---- CASE/event-level ----
        payload_cases     = payloads_df.select("CCDCaseReferenceNumberFrom").distinct().count()
        ack_cases         = ack_df.select("CCDCaseReferenceNumber").distinct().count()
        ack_success_cases = (ack_df.filter(upper(col("Status")) == "SUCCESS")
                             .select("CCDCaseReferenceNumber").distinct().count())

        # ---- printed funnel report ----
        report = [
            "",
            "================= CASE LINKING COUNT FUNNEL =================",
            " LINKS",
            f"   1. Expected links from silver  (sum n(n-1)/2)   : {expected_links:,}",
            f"   2. case_link_combinations rows                  : {comb_total:,}",
            f"        delta 1->2 (same-triple collapse, expected): {expected_links - comb_total:,}",
            f"   3. combinations with both CCD refs resolved     : {comb_resolved:,}",
            f"        delta 2->3 (cases not migrated, expected)  : {comb_total - comb_resolved:,}",
            f"   4. payload entries (sum of caseLinks)           : {payload_links:,}",
            f"   5. ack links (sum of terminal CaseLinkCount)    : {ack_links:,}",
            " CASES / EVENTS",
            f"   6. distinct payload 'from' cases                : {payload_cases:,}",
            f"   7. distinct ack cases                           : {ack_cases:,}",
            f"   8. distinct ack SUCCESS cases                   : {ack_success_cases:,}",
            " HARD CHECKS (nothing lost)",
            f"   3 == 4 (resolved combos -> payload links)       : {'OK' if comb_resolved == payload_links else 'MISMATCH'}",
            f"   4 == 5 (payload links  -> ack links)            : {'OK' if payload_links == ack_links else 'MISMATCH'}",
            f"   6 == 7 (payload cases  -> ack cases)            : {'OK' if payload_cases == ack_cases else 'MISMATCH'}",
            "============================================================",
            "",
        ]
        print("\n".join(report))

        # ---- record each stage count as an (informational) PASS result row ----
        def rec(field, value):
            results.append(TestResult(field, "PASS", f"count = {value}", state_under_test, fn))

        rec("case_link_e2e.count.1_expected_links_silver", expected_links)
        rec("case_link_e2e.count.2_combinations_rows", comb_total)
        rec("case_link_e2e.count.3_combinations_resolved", comb_resolved)
        rec("case_link_e2e.count.4_payload_links", payload_links)
        rec("case_link_e2e.count.5_ack_links", ack_links)
        rec("case_link_e2e.count.6_payload_cases", payload_cases)
        rec("case_link_e2e.count.7_ack_cases", ack_cases)
        rec("case_link_e2e.count.8_ack_success_cases", ack_success_cases)
        rec("case_link_e2e.count.delta_silver_to_combinations", expected_links - comb_total)
        rec("case_link_e2e.count.delta_combinations_unresolved", comb_total - comb_resolved)

        # ---- hard equality checks (FAIL if a stage lost links/cases) ----
        def eq(field, a, a_label, b, b_label):
            ok = (a == b)
            results.append(TestResult(
                field, "PASS" if ok else "FAIL",
                f"expected: {a_label} == {b_label} | actual: {a:,} vs {b:,}",
                state_under_test, fn))

        eq("case_link_e2e.count_resolved_eq_payload_links",
           comb_resolved, "resolved combinations", payload_links, "payload links")
        eq("case_link_e2e.count_payload_links_eq_ack_links",
           payload_links, "payload links", ack_links, "ack links")
        eq("case_link_e2e.count_payload_cases_eq_ack_cases",
           payload_cases, "payload cases", ack_cases, "ack cases")
    except Exception as e:
        results.append(TestResult("case_link_e2e.count_funnel", "FAIL", f"Test crashed: {e}", state_under_test, fn))
    return results


all_test_results.extend(test_count_funnel(silver_case_link_groups, case_link_combinations, case_link_payloads, ack_audit))

# Per-case link evidence - expected vs actual links (sample or all)

In [0]:
# Per-case expected vs actual link counts, as detailed evidence.
# Controlled by evidence_sample_size (None or 0 = ALL cases). Mismatches listed first.
_expected = case_link_payloads.select(
    col("CCDCaseReferenceNumberFrom").alias("ccd_ref"),
    size("caseLinks").alias("expected_links"),
)

_ack_term = ack_audit.groupBy("CCDCaseReferenceNumber").agg(
    smax(col("CaseLinkCount").cast("int")).alias("actual_links"),
    smax(when(upper(col("Status")) == "SUCCESS", 1).otherwise(0)).alias("has_success"),
)

_ack_err = (
    ack_audit.filter(upper(col("Status")) != "SUCCESS")
             .groupBy("CCDCaseReferenceNumber")
             .agg(F.first("Error", ignorenulls=True).alias("error"))
)

evidence_df = (
    _expected.join(_ack_term, _expected.ccd_ref == _ack_term.CCDCaseReferenceNumber, "left")
             .join(_ack_err,  _expected.ccd_ref == _ack_err.CCDCaseReferenceNumber, "left")
             .select(
                 col("ccd_ref"),
                 col("expected_links"),
                 F.coalesce(col("actual_links"), F.lit(0)).alias("actual_links"),
                 F.when(col("has_success") == 1, F.lit("SUCCESS")).otherwise(F.lit("FAILED/NO_ACK")).alias("terminal_status"),
                 col("error"),
             )
             .withColumn("match", col("expected_links") == col("actual_links"))
             .orderBy(col("match").asc(), col("ccd_ref"))   # mismatches first
)

total_cases = evidence_df.count()
if evidence_sample_size:
    evidence_rows = evidence_df.limit(int(evidence_sample_size)).collect()
    scope = f"{min(int(evidence_sample_size), total_cases)} of {total_cases}"
else:
    evidence_rows = evidence_df.collect()
    scope = f"ALL {total_cases}"

print(f"--- Per-case link evidence ({scope} cases; mismatches first) ---")
for r in evidence_rows:
    flag = "PASS" if r.match else "FAIL"
    err = f" | error: {(r.error or '')[:500]}" if not r.match else ""
    print(f"  [{flag}] {r.ccd_ref}: expected={r.expected_links} actual={r.actual_links} status={r.terminal_status}{err}")

# Rich table view (Databricks)
try:
    display(evidence_df if not evidence_sample_size else evidence_df.limit(int(evidence_sample_size)))
except Exception:
    pass

# Push evidence rows into the test results so they persist in the CSV + central table
for r in evidence_rows:
    msg = f"expected: {r.expected_links} links | actual: {r.actual_links} links | status: {r.terminal_status}"
    if not r.match:
        msg += f" | error: {(r.error or '')[:4000]}"
    all_test_results.append(TestResult(
        f"case_link_e2e.evidence.{r.ccd_ref}",
        "PASS" if r.match else "FAIL",
        msg,
        state_under_test,
        "test_link_evidence",
    ))

# Blocked-link breakdown - why each link could not be created (categorised)

In [0]:
def test_blocked_link_breakdown(combinations_df):
    """Categorise every link that cannot be created because a participating case was not
    migrated to CCD (blocked upstream, e.g. DQ). Three shapes:
      BOTH_BLOCKED              - neither case in CCD; link cannot be created.
      MIGRATED_FROM_INCOMPLETE  - From is in CCD (succeeded) but To blocked; the migrated
                                  From case's payload is silently missing this link.
      MIGRATED_TO_INCOMPLETE    - To is in CCD but From (older originator) blocked; the
                                  migrated To case never gets this 'Linked From' relationship.
    The two MIGRATED_* shapes are the dangerous ones - those cases show SUCCESS yet are
    actually incomplete, and nothing else in the success path flags them."""
    fn = inspect.stack()[0].function
    results = []
    try:
        dropped = combinations_df.filter(
            col("CCDCaseReferenceNumberFrom").isNull() | col("CCDCaseReferenceNumberTo").isNull()
        ).withColumn(
            "category",
            when(col("CCDCaseReferenceNumberFrom").isNull() & col("CCDCaseReferenceNumberTo").isNull(),
                 F.lit("BOTH_BLOCKED"))
            .when(col("CCDCaseReferenceNumberFrom").isNotNull() & col("CCDCaseReferenceNumberTo").isNull(),
                 F.lit("MIGRATED_FROM_INCOMPLETE"))
            .otherwise(F.lit("MIGRATED_TO_INCOMPLETE"))
        )

        total = dropped.count()
        cat = {r["category"]: r["c"] for r in dropped.groupBy("category").agg(F.count("*").alias("c")).collect()}
        both     = cat.get("BOTH_BLOCKED", 0)
        mig_from = cat.get("MIGRATED_FROM_INCOMPLETE", 0)
        mig_to   = cat.get("MIGRATED_TO_INCOMPLETE", 0)

        results.append(TestResult("case_link_e2e.links_dropped_total",
            "FAIL" if total else "PASS",
            f"{total} links cannot be created (a participating case is not in CCD)",
            state_under_test, fn))
        results.append(TestResult("case_link_e2e.dropped_both_blocked",
            "FAIL" if both else "PASS",
            f"BOTH_BLOCKED (neither case in CCD): {both} links",
            state_under_test, fn))
        results.append(TestResult("case_link_e2e.dropped_migrated_from_incomplete",
            "FAIL" if mig_from else "PASS",
            f"MIGRATED_FROM_INCOMPLETE (From in CCD succeeded but payload missing a link to a blocked To): {mig_from} links",
            state_under_test, fn))
        results.append(TestResult("case_link_e2e.dropped_migrated_to_incomplete",
            "FAIL" if mig_to else "PASS",
            f"MIGRATED_TO_INCOMPLETE (To in CCD but blocked originator, so no inbound link created): {mig_to} links",
            state_under_test, fn))

        # distinct migrated cases that are incomplete - the priority list
        from_inc = dropped.filter(col("category") == "MIGRATED_FROM_INCOMPLETE").select(col("CaseNoFrom").alias("CaseNo")).distinct()
        to_inc   = dropped.filter(col("category") == "MIGRATED_TO_INCOMPLETE").select(col("CaseNoTo").alias("CaseNo")).distinct()
        migrated_incomplete = from_inc.unionByName(to_inc).distinct()
        mi_n = migrated_incomplete.count()
        msg = f"migrated cases that are in CCD but have INCOMPLETE links (need re-link once partner DQ fixed): {mi_n}"
        if mi_n:
            msg += f" (sample: {[r.CaseNo for r in migrated_incomplete.limit(10).collect()]})"
        results.append(TestResult("case_link_e2e.migrated_cases_incomplete_links",
            "FAIL" if mi_n else "PASS", msg, state_under_test, fn))

        # per-link evidence rows with category (bounded by evidence_sample_size; None/0 = all)
        detail = dropped.select("CaseNoFrom", "CaseNoTo", "ReasonLinkId", "category")
        ev = detail.limit(int(evidence_sample_size)).collect() if evidence_sample_size else detail.collect()
        for r in ev:
            all_test_results.append(TestResult(
                f"case_link_e2e.dropped_link.{r.CaseNoFrom}->{r.CaseNoTo}",
                "FAIL",
                f"category: {r.category} | reason {r.ReasonLinkId}",
                state_under_test, "test_blocked_link_breakdown"))

        print("--- Blocked-link breakdown ---")
        print(f"  total links not created            : {total}")
        print(f"  BOTH_BLOCKED                       : {both}")
        print(f"  MIGRATED_FROM_INCOMPLETE           : {mig_from}")
        print(f"  MIGRATED_TO_INCOMPLETE             : {mig_to}")
        print(f"  distinct migrated cases incomplete : {mi_n}")

        # --- Consolidated silent-failures table: in-CCD cases with a MISSING link ---
        # (MIGRATED_* only; BOTH_BLOCKED have no in-CCD case to flag). Direction shows
        # whether the migrated case is missing an OUTBOUND link (partner To blocked) or
        # an INBOUND link (originator From blocked).
        silent = (
            dropped.filter(col("category") != "BOTH_BLOCKED")
                .withColumn("in_ccd_case",
                    when(col("category") == "MIGRATED_FROM_INCOMPLETE", col("CaseNoFrom")).otherwise(col("CaseNoTo")))
                .withColumn("blocked_partner",
                    when(col("category") == "MIGRATED_FROM_INCOMPLETE", col("CaseNoTo")).otherwise(col("CaseNoFrom")))
                .select("in_ccd_case", "blocked_partner", "ReasonLinkId", "category")
                .orderBy("in_ccd_case", "blocked_partner")
        )
        silent_rows = silent.collect()
        distinct_cases = len({r.in_ccd_case for r in silent_rows})

        print("--- Silent failures: in-CCD cases with a MISSING link (re-link once partner DQ fixed) ---")
        print(f"  {len(silent_rows)} missing links across {distinct_cases} distinct in-CCD cases")
        if silent_rows:
            print(f"  {'CaseNo (in CCD)':<18}{'Direction':<10}{'Blocked partner':<18}{'Reason':<7}")
            for r in silent_rows:
                dirn = "OUTBOUND" if r.category == "MIGRATED_FROM_INCOMPLETE" else "INBOUND"
                print(f"  {r.in_ccd_case:<18}{dirn:<10}{r.blocked_partner:<18}{str(r.ReasonLinkId):<7}")

        # record one result row per silent-failure link
        for r in silent_rows:
            dirn = ("OUTBOUND - missing link TO blocked partner"
                    if r.category == "MIGRATED_FROM_INCOMPLETE"
                    else "INBOUND - missing link FROM blocked partner")
            results.append(TestResult(
                f"case_link_e2e.silent_failure.{r.in_ccd_case}->{r.blocked_partner}",
                "FAIL",
                f"{dirn} | blocked partner: {r.blocked_partner} | reason {r.ReasonLinkId} | {r.category}",
                state_under_test, fn))

        # --- Both-blocked links: neither case in CCD, link cannot exist (lost entirely) ---
        both_df = (
            dropped.filter(col("category") == "BOTH_BLOCKED")
                .select("CaseNoFrom", "CaseNoTo", "ReasonLinkId")
                .orderBy("CaseNoFrom", "CaseNoTo")
        )
        both_rows = both_df.collect()
        print("--- Both-blocked links: neither case in CCD (link cannot be created, no error raised) ---")
        print(f"  {len(both_rows)} links lost entirely")
        if both_rows:
            print(f"  {'From (blocked)':<18}{'To (blocked)':<18}{'Reason':<7}")
            for r in both_rows:
                print(f"  {r.CaseNoFrom:<18}{r.CaseNoTo:<18}{str(r.ReasonLinkId):<7}")

        # record one result row per both-blocked link
        for r in both_rows:
            results.append(TestResult(
                f"case_link_e2e.both_blocked.{r.CaseNoFrom}->{r.CaseNoTo}",
                "FAIL",
                f"BOTH_BLOCKED - neither case in CCD | From: {r.CaseNoFrom} | To: {r.CaseNoTo} | reason {r.ReasonLinkId}",
                state_under_test, fn))
    except Exception as e:
        results.append(TestResult("case_link_e2e.blocked_link_breakdown", "FAIL", f"Test crashed: {e}", state_under_test, fn))
    return results


all_test_results.extend(test_blocked_link_breakdown(case_link_combinations))

In [0]:
# --- 422 duplicate-top-level-id failures: caseno, CCD ref, and how badly duplicated ---
_mappings_422 = spark.table("hive_metastore.ariadm_active_appeals.aria_ccd_case_mappings")

# every case whose terminal ack carries the dup-id 422
_failed_422 = (
    ack_audit.groupBy("CCDCaseReferenceNumber")
        .agg(smax(when(upper(col("Status")) == "SUCCESS", 1).otherwise(0)).alias("ok"),
             F.first("Error", ignorenulls=True).alias("error"))
        .filter("ok = 0")
        .filter(col("error").contains("Collection item ID must be unique"))
)

# Split CURRENT failures (case still has a payload in this gold) from STALE orphan acks
# (422 in the audit but no payload now - e.g. a case whose self-pair was removed in a gold
#  rebuild, so its old ack lingers but it is no longer sent and would not 422 again).
_payload_refs = case_link_payloads.select(col("CCDCaseReferenceNumberFrom").alias("p_ref")).distinct()
_current_422 = _failed_422.join(_payload_refs, _failed_422.CCDCaseReferenceNumber == _payload_refs.p_ref, "leftsemi")
_stale_422   = _failed_422.join(_payload_refs, _failed_422.CCDCaseReferenceNumber == _payload_refs.p_ref, "leftanti")

# Duplication magnitude per case, straight from the payload: total caseLinks entries,
# distinct linked-to refs, and the excess (dupes) that cause the 422.
_counts = (
    case_link_payloads
        .select(col("CCDCaseReferenceNumberFrom").alias("ref"), F.explode("caseLinks").alias("cl"))
        .select("ref", col("cl.id").alias("to_id"))
        .groupBy("ref")
        .agg(F.count("*").alias("links"), F.countDistinct("to_id").alias("distinct_links"))
        .withColumn("dupes", col("links") - col("distinct_links"))
)

_cur = (
    _current_422
        .join(_mappings_422, _current_422.CCDCaseReferenceNumber == _mappings_422.CCDCaseID, "left")
        .join(_counts, _current_422.CCDCaseReferenceNumber == _counts.ref, "left")
        .select(col("CaseNo").alias("caseno"),
                col("CCDCaseReferenceNumber").alias("ccd_ref"),
                F.coalesce(col("links"), F.lit(0)).alias("links"),
                F.coalesce(col("distinct_links"), F.lit(0)).alias("distinct_links"),
                F.coalesce(col("dupes"), F.lit(0)).alias("dupes"))
        .orderBy(col("dupes").desc(), "caseno")
)
_table_422 = _cur.collect()

_stale = (
    _stale_422.join(_mappings_422, _stale_422.CCDCaseReferenceNumber == _mappings_422.CCDCaseID, "left")
        .select(col("CaseNo").alias("caseno"), col("CCDCaseReferenceNumber").alias("ccd_ref"))
        .orderBy("caseno")
        .collect()
)

print(f"--- 422 duplicate-top-level-id failures (CURRENT gold): {len(_table_422)} cases ---")
print(f"  {'CaseNo':<16}{'CCD Reference':<20}{'Links':<7}{'Distinct':<10}{'Dupes':<7}")
for r in _table_422:
    print(f"  {(r.caseno or '?'):<16}{r.ccd_ref:<20}{r.links:<7}{r.distinct_links:<10}{r.dupes:<7}")

if _stale:
    print(f"\n  Excluded {len(_stale)} STALE orphan ack(s): 422 in audit but no payload in current gold")
    print(f"  (already resolved - would not be re-sent):")
    for r in _stale:
        print(f"    {(r.caseno or '?'):<16}{r.ccd_ref}")

# record to central results: summary + one row per CURRENT failure
all_test_results.append(TestResult(
    "case_link_e2e.dup_id_422_total",
    "FAIL" if _table_422 else "PASS",
    f"{len(_table_422)} CURRENT cases rejected by CCD with 422 'Collection item ID must be unique'"
    + (f" | {len(_stale)} stale orphan ack(s) excluded: {[(r.caseno or r.ccd_ref) for r in _stale]}" if _stale else ""),
    state_under_test, "test_422_failures"))
for r in _table_422:
    all_test_results.append(TestResult(
        f"case_link_e2e.dup_id_422.{r.caseno or r.ccd_ref}",
        "FAIL",
        f"ccd_ref: {r.ccd_ref} | links: {r.links} | distinct: {r.distinct_links} | duplicates: {r.dupes}",
        state_under_test, "test_422_failures"))

# rich table + CSV for the evidence pack
try:
    display(_cur.withColumnRenamed("distinct_links", "distinct").withColumnRenamed("dupes", "duplicates"))
    _cur.toPandas().to_csv(test_results_path + "/422_cases.csv", index=False)
    print("Saved: " + test_results_path + "/422_cases.csv")
except Exception as e:
    print(f"(display/csv skipped: {e})")


In [0]:
elapsed = perf_time.time() - t0
counts = count_by_status(all_test_results)

print(f"Tests complete in {elapsed:.1f}s")
print(f"  Total : {counts['TOTAL']}")
print(f"  Pass  : {counts['PASS']}")
print(f"  Fail  : {counts['FAIL']}")
print(f"  NoData: {counts['NO_DATA']}")
print(f"  Error : {counts['ERROR']}")

## Write reports + audit

In [0]:
status_order = {"FAIL": 0, "ERROR": 1, "NO_DATA": 2, "PASS": 3}
all_test_results.sort(key=lambda r: (status_order.get(str(r.status).upper(), 4), str(r.test_field)))

folder, timestamp, _ = build_report_folder(test_results_path, f"{state_under_test}_{run_tag}")
print(f"Reports folder: {folder}")

try:
    csv_path = write_csv(all_test_results, state_under_test, folder, timestamp)
    print(f"CSV  : {csv_path}")
except Exception as e:
    print(f"write_csv FAILED: {e}")

try:
    run_id = create_run(
        spark,
        run_user=run_user,
        run_start_datetime=run_start_datetime,
        run_by_automation_name="Case_Linking_Tests",
        run_tag=run_tag,
        run_status=("FAILED" if any(str(getattr(r, "status", "")).upper().strip() in ("FAIL", "ERROR") for r in all_test_results) else "PASSED"),
        state_under_test=state_under_test,
    )
    print(f"Created run_id={run_id}")
    n = create_results(spark, run_id, all_test_results)
    print(f"Wrote {n} result rows")
except Exception as e:
    print(f"audit write FAILED: {e}")